### f1: lista dei numeri da 0 a n-1
- Il `for` è stato sostituito da un `while` per avere un contatore esplicito su cui scrivere l'invariante.
- Senza `Requires(n >= 0)` la verifica fallisce con: *Loop invariant might not hold on entry. Assertion (i <= n) might not hold.* L'errore non è sulla postcondizione ma sull'ingresso nel ciclo: Nagini segnala il primo punto della dimostrazione che non regge.

#### Cosa controlla Nagini
* **Statement dimostrato:** per ogni n ≥ 0, len(f1(n)) = n.

1. **Ingresso:** i = 0, l vuota, quindi len(l) = i; i ≤ n vale grazie a n ≥ 0.
2. **Conservazione:** `append` e `i += 1` aumentano entrambi di 1.
3. **Uscita:** guardia falsa (i ≥ n) e invariante (i ≤ n) danno i = n, quindi len(l) = n.

#### Risultato 
* Verification successful


In [ ]:
from typing import List
from nagini_contracts.contracts import *

def f1(n: int) -> List[int]:
    Requires(n >= 0) #precondizione
    Ensures(list_pred(Result())) #permesso sulla lista
    Ensures(len(Result()) == n) #postcondizione , ciò che vogliamo dimostrare
    l: List[int] = []
    i = 0
    while i < n:
        #invariante vera per ogni iterazione
        Invariant(list_pred(l)) # senza questo append fallirebbe per mancanza di permessi
        Invariant(i <= n)
        Invariant(len(l) == i) #punto fondamentale per la dimostrazione della postcondizione
        l.append(i)
        i += 1
    return l

### nFibonacci: primi n numeri della sequenza di Fibonacci
- Il `for` è stato sostituito da un `while` per avere un contatore esplicito su cui scrivere l'invariante.
- il contratto e le invarianti solo le stesse di f1
- **Importante** 
    - gli invarianti non parlano di a e b. Per la lunghezza i valori sono irrilevanti, e infatti la dimostrazione è identica a quella di f1.
    - Nagini non ha bisogno di sapere niente sul contenuto della lista per dimostrarne la lunghezza.

#### Come ragiona Nagini (induzione sulle iterazioni)
* **Statement dimostrato:** per ogni n ≥ 0, len(nFibonacci(n)) = n.

* i controlli di nagini sono gli stessi che fa per f1

#### Risultato
* Verification successful

In [ ]:
def nFibonacci(n: int) -> List[int]:
    Requires(n >= 0)
    Ensures(list_pred(Result()))
    Ensures(len(Result()) == n)
    f: List[int] = []
    a, b = 0, 1
    i = 0
    while i < n:
        Invariant(list_pred(f))
        Invariant(i <= n)
        Invariant(len(f) == i)
        f.append(a)
        a, b = b, a + b #assegnazione simultanea permessa da nagini
        i += 1
    return f


## triangolo: [[0], [0, 1], [0, 1, 2], ...]
- Il `for` esterno è stato sostituito da un `while` per avere un contatore esplicito su cui scrivere l'invariante.
- Le righe sono oggetti separati dalla lista esterna: serve un permesso quantificato, `Forall(l, lambda r: list_pred(r))`, per leggerle e modificarle.
- La lunghezza delle righe si specifica con un quantificatore sugli indici, `Forall(int, ...)`, limitato con `Implies` agli indici validi.
- rispetto ad f1 la difficoltà sta nel gestire i permessi su oggetti annidati

### Cosa controlla Nagini
* **Statement dimostrati:** per ogni n ≥ 0, l'output ha n righe e la riga k ha k+1 elementi.

**Ciclo interno** -> stesso schema di f1, con i+1 al posto di n

**Ciclo esterno**
1. **Ingresso:** i = 0, l vuota; il `Forall` sulle righe vale perché non ci sono righe.
2. **Conservazione:**
   - le righe già inserite non cambiano: i permessi pieni su `riga` e sulle righe di `l` garantiscono che `riga` non sia una di esse (niente aliasing);
   - la nuova riga ha lunghezza i+1, quindi il `Forall` vale fino a k < i+1;
   - dopo l'`append`, il permesso su `riga` entra nel permesso quantificato su `l`.
3. **Uscita:** da i ≥ n e i ≤ n segue i = n; quindi n righe, la riga k di lunghezza k+1.

---
## triangolo: tentativi di verifica

### 1. Versione iniziale
- Postcondizioni sulle righe scritte con `Forall(Result(), lambda r: list_pred(r))`.
- **Risultato:** *Type error: Encountered Any type* (riga 7).
- **Causa:** `Result()` ha tipo `Any`, quindi anche il parametro `r` della lambda risulta `Any`, e Nagini lo rifiuta.

### 2. `cast` sul risultato
- `Forall(cast(List[List[int]], Result()), ...)`.
- **Risultato:** prima *Name "cast" is not defined* (mancava l'import), poi *invalid.forall* (riga 7).
- **Causa:** Nagini non accetta una chiamata a `cast` come dominio di un `Forall`.

### 3. Statement sulle righe spostato in un `Assert` prima del `return`
- Eliminati i `Forall` su `Result()` dalle postcondizioni.
- **Risultato:** *invalid.forall* (riga 12, invariante `Forall(l, lambda r: list_pred(r))`).
- **Conclusione:** il problema non era `Result()`. Nagini rifiuta il `Forall` sugli elementi di una lista combinato con `list_pred`.

### 4. Permesso sulle righe quantificato sugli indici
- `Forall(int, lambda k: Implies(0 <= k and k < len(l), list_pred(l[k])))`.
- **Risultato:** traduzione riuscita; verifica fallita con *The precondition of function len might not hold. There might be insufficient permission* (riga 15, su `len(l[k])`).
- **Conclusione:** la forma è accettata, ma il permesso non risulta sufficiente per leggere la lunghezza delle righe.

### 5. Invarianti esterni ripetuti nel ciclo interno
- Ipotesi: i permessi su `l` si perdono attraverso il ciclo interno.
- **Risultato:** stesso errore alla riga 15.
- **Conclusione:** ipotesi scartata.

### 6. `Assert` intermedi nel corpo del ciclo (A-D)
- Per localizzare dove si perde il permesso.
- **Risultato:** stesso errore alla riga 15; nessun assert tra gli errori.
- **Conclusione:** l'errore avviene prima del corpo del ciclo, cioè nella formulazione degli invarianti e non nella loro conservazione.

### 7. File `prova.py`: invarianti come precondizioni, senza cicli
- **prova1:** forma attuale (`k < i`).
- **prova2:** stessa condizione `k < len(l)` in entrambi i quantificatori.
- **prova3:** permesso scritto con `Acc(list_pred(l[k]))`.
- **prova4:** precondizioni ripetute come postcondizioni, per testare l'esalazione.
- **prova5:** permesso e lunghezza nello stesso quantificatore.
- **Risultato:** tutte e cinque falliscono con lo stesso errore su `len(l[k])`, già nelle precondizioni. (Un'esecuzione precedente di prova1-3 aveva dato successo, probabilmente su una versione del file non salvata.)
- **Conclusione:** il permesso dichiarato con `Forall(int, ... list_pred(l[k]))` non viene riconosciuto come sufficiente per `len(l[k])`, in nessuna delle varianti provate.

### Problemi d'ambiente incontrati
- Crash *'NoneType' object has no attribute 'path'*: risolto cancellando le cache di Nagini (`.mypy_cache_strict` e `.mypy_cache_nonstrict`) e rinominando `test.py`, che va in conflitto con il pacchetto `test` della libreria standard.
- *Incompatible import of "Assert"*: l'editor aveva aggiunto in automatico `from ast import Assert`, in conflitto con l'`Assert` di Nagini.

### Stato attuale
- Dimostrabile: `len(Result()) == n`, con la stessa struttura di f1.
- Aperto: lo statement sulle lunghezze delle righe. Prossimo passo: cercare nei test del repository di Nagini come vengono espressi i permessi su liste annidate.
- Considerazione: la difficoltà non sta nella proprietà, che è banale, ma nell'esprimere i permessi su oggetti annidati.


In [ ]:
def triangolo(n: int) -> List[List[int]]:
    Requires(n >= 0)
    Ensures(list_pred(Result()))
    Ensures(Forall(Result(), lambda r: list_pred(r)))
    Ensures(len(Result()) == n)
    Ensures(Forall(int, lambda k: Implies(0 <= k and k < n, len(Result()[k]) == k + 1)))
    l: List[List[int]] = []
    i = 0
    while i < n:
        Invariant(list_pred(l))
        Invariant(Forall(l, lambda r: list_pred(r)))
        Invariant(0 <= i and i <= n)
        Invariant(len(l) == i)
        Invariant(Forall(int, lambda k: Implies(0 <= k and k < i, len(l[k]) == k + 1)))
        riga: List[int] = []
        j = 0
        while j <= i:
            Invariant(list_pred(riga))
            Invariant(0 <= j and j <= i + 1)
            Invariant(len(riga) == j)
            riga.append(j)
            j += 1
        l.append(riga)
        i += 1
    return l

In [ ]:
def matrixI(n: int) -> List[List[int]]:
    Requires(n >= 0)
    Ensures(list_pred(Result()))
    Ensures(Forall(Result(), lambda r: list_pred(r)))
    Ensures(len(Result()) == n)
    Ensures(Forall(int, lambda k: Implies(0 <= k and k < n,
                                          len(Result()[k]) == n)))
    M: List[List[int]] = []
    i = 0
    while i < n:
        Invariant(list_pred(M))
        Invariant(Forall(M, lambda r: list_pred(r)))
        Invariant(0 <= i and i <= n)
        Invariant(len(M) == i)
        Invariant(Forall(int, lambda k: Implies(0 <= k and k < i,
                                                len(M[k]) == n)))
        riga: List[int] = []
        j = 0
        while j < n:
            Invariant(list_pred(riga))
            Invariant(0 <= j and j <= n)
            Invariant(len(riga) == j)
            riga.append(1 if i == j else 0)
            j += 1
        M.append(riga)
        i += 1
    return M

In [ ]:
# Funzione pura per esprimere 2^k nelle specifiche
@Pure
def pow2(k: int) -> int:
    Requires(k >= 0)
    return 1 if k == 0 else 2 * pow2(k - 1)

def sottoinsiemi(n: int) -> List[List[int]]:
    Requires(n >= 0)
    Ensures(list_pred(Result()))
    Ensures(Forall(Result(), lambda x: list_pred(x)))
    Ensures(len(Result()) == pow2(n))
    vuoto: List[int] = []
    s: List[List[int]] = [vuoto]
    k = 0
    while k < n:
        Invariant(list_pred(s))
        Invariant(Forall(s, lambda x: list_pred(x)))
        Invariant(0 <= k and k <= n)
        Invariant(len(s) == pow2(k))
        new: List[List[int]] = []
        j = 0
        while j < len(s):
            Invariant(list_pred(s))
            Invariant(Forall(s, lambda x: list_pred(x)))
            Invariant(len(s) == pow2(k))
            Invariant(list_pred(new))
            Invariant(Forall(new, lambda x: list_pred(x)))
            Invariant(0 <= j and j <= len(s))
            Invariant(len(new) == j)
            new.append(s[j] + [k])
            j += 1
        s = s + new
        k += 1
    return s